# Exploratory Data Analysis: CEEW Smart Meter Data
## Theme 8 - AI for Smart Meter Intelligence & Loss Detection

This notebook explores the CEEW smart meter dataset from Bareilly and Mathura (2019-2021).

**Objectives:**
1. Load and inspect raw data structure
2. Understand consumption patterns and seasonality
3. Identify data quality issues
4. Explore zone-level and meter-level characteristics
5. Validate data for model training

## 1. Import Required Libraries

In [15]:
import sys
import os
from pathlib import Path

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## 2. Set Up Project Environment

In [16]:
# Set up logging
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Define paths
PROJ_ROOT = Path.cwd().parent
DATA_RAW = PROJ_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJ_ROOT / 'data' / 'processed'
OUTPUTS = PROJ_ROOT / 'outputs'

# Create directories if they don't exist
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {PROJ_ROOT}")
print(f"Data Raw: {DATA_RAW}")
print(f"Data Processed: {DATA_PROCESSED}")
print(f"Outputs: {OUTPUTS}")

Project Root: c:\Users\Lenovo\Desktop\aiforBharat\theme8
Data Raw: c:\Users\Lenovo\Desktop\aiforBharat\theme8\data\raw
Data Processed: c:\Users\Lenovo\Desktop\aiforBharat\theme8\data\processed
Outputs: c:\Users\Lenovo\Desktop\aiforBharat\theme8\outputs


## 3. Initialize Data Structures - Load Data

In [17]:
# Import data loader
from src.data.data_loader import DataLoader

# Initialize loader
loader = DataLoader(data_dir=str(DATA_RAW))

# Check available files
print("Checking data files...")
file_status = loader.get_file_status()
for file, exists in file_status.items():
    status = '✓' if exists else '✗'
    print(f"  {status} {file}")

2026-05-04 12:10:29,635 - src.data.data_loader - INFO - DataLoader initialized with data_dir: c:\Users\Lenovo\Desktop\aiforBharat\theme8\data\raw


Checking data files...
  ✗ CEEW - Smart meter data Bareilly 2019.csv
  ✓ CEEW - Smart meter data Bareilly 2020.csv
  ✓ CEEW - Smart meter data Bareilly 2021.csv
  ✓ CEEW - Smart meter data Mathura 2019.csv
  ✓ CEEW - Smart meter data Mathura 2020.csv
  ✗ CEEW - Smart meter data Mathura 2021.csv


In [18]:
# Load raw data
print("Loading raw data...")
raw_data = loader.load_raw_data()

print(f"\nLoaded data summary:")
for zone, df in raw_data.items():
    print(f"  {zone.capitalize()}: {len(df)} rows")

2026-05-04 12:10:29,659 - src.data.data_loader - WARNING -   ✗ File not found: c:\Users\Lenovo\Desktop\aiforBharat\theme8\data\raw\CEEW - Smart meter data Bareilly 2019.csv
2026-05-04 12:10:29,662 - src.data.data_loader - INFO - Loading CEEW - Smart meter data Bareilly 2020.csv...


Loading raw data...


2026-05-04 12:10:37,578 - src.data.data_loader - INFO -   ✓ Loaded 6627360 rows
2026-05-04 12:10:37,582 - src.data.data_loader - INFO - Loading CEEW - Smart meter data Bareilly 2021.csv...
2026-05-04 12:10:42,902 - src.data.data_loader - INFO -   ✓ Loaded 3948960 rows
2026-05-04 12:10:43,576 - src.data.data_loader - INFO - Bareilly data: 10576320 total rows
2026-05-04 12:10:43,584 - src.data.data_loader - INFO - Loading CEEW - Smart meter data Mathura 2019.csv...
2026-05-04 12:10:49,139 - src.data.data_loader - INFO -   ✓ Loaded 3588874 rows
2026-05-04 12:10:49,235 - src.data.data_loader - INFO - Loading CEEW - Smart meter data Mathura 2020.csv...
2026-05-04 12:10:55,779 - src.data.data_loader - INFO -   ✓ Loaded 3759360 rows
2026-05-04 12:10:55,821 - src.data.data_loader - WARNING -   ✗ File not found: c:\Users\Lenovo\Desktop\aiforBharat\theme8\data\raw\CEEW - Smart meter data Mathura 2021.csv
2026-05-04 12:10:56,290 - src.data.data_loader - INFO - Mathura data: 7348234 total rows



Loaded data summary:
  Bareilly: 10576320 rows
  Mathura: 7348234 rows


In [19]:
# Combine data
print("Combining Bareilly and Mathura data...")
df_combined = loader.combine_data(raw_data)

# Standardize column names
df_combined = loader.standardize_column_names(df_combined)

print(f"Combined dataset: {df_combined.shape}")
print(f"\nColumns: {df_combined.columns.tolist()}")

Combining Bareilly and Mathura data...


2026-05-04 12:11:51,583 - src.data.data_loader - INFO - Combined data: 17924554 rows, 7 columns
2026-05-04 12:11:59,810 - src.data.data_loader - INFO - Zones: ['Bareilly' 'Mathura']
2026-05-04 12:12:34,312 - src.data.data_loader - INFO - Standardized column names. Original: 7 → 7


Combined dataset: (17924554, 7)

Columns: ['timestamp', 'kwh', 'avg_voltage_volt', 'avg_current_amp', 'freq_hz', 'meter', 'zone']


In [20]:
import os

dataset_path = DATA_RAW

print("Dataset path:", dataset_path)
print(os.listdir(dataset_path))

Dataset path: c:\Users\Lenovo\Desktop\aiforBharat\theme8\data\raw
['CEEW - Smart meter data Bareilly 2020.csv', 'CEEW - Smart meter data Bareilly 2021.csv', 'CEEW - Smart meter data Mathura 2019.csv', 'CEEW - Smart meter data Mathura 2020.csv', 'SM Cleaned Data BR Aggregated.csv', 'SM Cleaned Data BR2019.csv', 'SM Cleaned Data MH Aggregated.csv', 'SM Cleaned Data MH2021.csv']


## 4. Implement Core Functions - Data Inspection

In [21]:
# Display data structure
print("=" * 60)
print("DATA STRUCTURE")
print("=" * 60)
print(f"\nShape: {df_combined.shape}")
print(f"\nData Types:")
print(df_combined.dtypes)
print(f"\nFirst 5 rows:")
df_combined.head()

DATA STRUCTURE

Shape: (17924554, 7)

Data Types:
timestamp            object
kwh                 float64
avg_voltage_volt    float64
avg_current_amp     float64
freq_hz             float64
meter                object
zone                 object
dtype: object

First 5 rows:


,timestamp,kwh,avg_voltage_volt,avg_current_amp,freq_hz,meter,zone
0,2020-01-01 00:00:00,0.002,251.26,0.15,49.97,BR02,Bareilly
1,2020-01-01 00:03:00,0.001,251.23,0.15,49.94,BR02,Bareilly
2,2020-01-01 00:06:00,0.001,251.55,0.14,49.94,BR02,Bareilly
3,2020-01-01 00:09:00,0.001,251.97,0.14,50.09,BR02,Bareilly
4,2020-01-01 00:12:00,0.002,252.03,0.14,50.08,BR02,Bareilly


In [22]:
# Check for missing values
print("=" * 60)
print("MISSING VALUES")
print("=" * 60)
missing = df_combined.isnull().sum()
missing_pct = (missing / len(df_combined) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Count'] > 0])

MISSING VALUES
Empty DataFrame
Columns: [Count, Percentage]
Index: []


In [23]:
# Standardize timestamp column
from src.data.data_processor import DataProcessor

# Find timestamp column
timestamp_cols = [col for col in df_combined.columns if 'time' in col.lower()]
print(f"Timestamp columns found: {timestamp_cols}")

if timestamp_cols:
    timestamp_col = timestamp_cols[0]
    print(f"Using: {timestamp_col}")
    
    # Create processor
    processor = DataProcessor(df_combined, timestamp_col=timestamp_col)
    processor.standardize_timestamps()
    df_combined = processor.df
    
    print(f"\nTimestamp range:")
    print(f"  Start: {df_combined[timestamp_col].min()}")
    print(f"  End: {df_combined[timestamp_col].max()}")
    print(f"  Duration: {(df_combined[timestamp_col].max() - df_combined[timestamp_col].min()).days} days")

Timestamp columns found: ['timestamp']
Using: timestamp


2026-05-04 12:12:59,361 - src.data.data_processor - INFO - DataProcessor initialized with 17924554 rows
2026-05-04 12:12:59,411 - src.data.data_processor - INFO - Standardizing timestamps...
2026-05-04 12:13:08,394 - src.data.data_processor - INFO -   ✓ Parsed timestamp



Timestamp range:
  Start: 2019-05-01 00:00:00
  End: 2021-10-31 23:57:00
  Duration: 914 days


In [24]:
# Analyze consumption column
consumption_cols = [col for col in df_combined.columns if 'kwh' in col.lower()]
print(f"Consumption columns: {consumption_cols}")

if consumption_cols:
    consumption_col = consumption_cols[0]
    print(f"\nUsing: {consumption_col}")
    print(f"\nConsumption Statistics:")
    print(df_combined[consumption_col].describe())

Consumption columns: ['kwh']

Using: kwh

Consumption Statistics:
count    1.792455e+07
mean     1.775483e-02
std      2.552322e-02
min      0.000000e+00
25%      3.000000e-03
50%      1.000000e-02
75%      2.100000e-02
max      3.000000e-01
Name: kwh, dtype: float64


## 5. Test Implementation - Data Quality Checks

In [25]:
import plotly.io as pio
pio.renderers.default = "browser"


In [29]:
# Visualize consumption distribution by zone
df_sample  = df_combined.sample(1000, random_state=42)  # Sample for faster plotting

fig = px.box(df_sample, x="zone", y=consumption_col)
fig.show()

In [34]:
import plotly.express as px
import pandas as pd

# Ensure timestamp is datetime
df_sample[timestamp_col] = pd.to_datetime(df_sample[timestamp_col])

# Daily aggregation by zone
daily_agg = (
    df_sample
    .groupby([pd.Grouper(key=timestamp_col, freq='D'), 'zone'])[consumption_col]
    .sum()
    .unstack('zone')
    .fillna(0)
)

# Plot
fig = px.line(
    daily_agg,
    x=daily_agg.index,
    y=daily_agg.columns,
    title="Daily Consumption Trends by Zone",
    labels={"value": "Consumption (kWh)", "index": "Date"}
)

fig.update_layout(xaxis_title="Date", yaxis_title="Consumption (kWh)")
fig.show()

In [ ]:
# Verify data quality
print("=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)

# Check for duplicates
duplicates = df_combined.duplicated().sum()
print(f"\n1. Duplicates: {duplicates} rows")

# Check for zero/negative consumption
if consumption_col in df_combined.columns:
    zero_count = (df_combined[consumption_col] == 0).sum()
    negative_count = (df_combined[consumption_col] < 0).sum()
    print(f"\n2. Consumption values:")
    print(f"   Zero: {zero_count}")
    print(f"   Negative: {negative_count}")

# Check zones
print(f"\n3. Zones: {df_combined['zone'].nunique()} unique zones")
print(df_combined['zone'].value_counts())

print(f"\n✓ Data quality checks complete")

## 6. Optimize Performance - Summary Statistics

In [35]:
# Generate summary report
print("=" * 60)
print("EDA SUMMARY REPORT")
print("=" * 60)

summary = {
    'Total Records': len(df_combined),
    'Date Range': f"{df_combined[timestamp_col].min()} to {df_combined[timestamp_col].max()}",
    'Duration (days)': (df_combined[timestamp_col].max() - df_combined[timestamp_col].min()).days,
    'Zones': df_combined['zone'].nunique(),
    'Missing Values': df_combined.isnull().sum().sum(),
    'Duplicates': df_combined.duplicated().sum(),
    'Memory (MB)': df_combined.memory_usage(deep=True).sum() / 1024**2
}

for key, value in summary.items():
    print(f"{key:.<40} {value}")

print("\n✓ EDA Complete!")
print("\nNext Steps:")
print("1. Run 02_demand_forecasting_dev.ipynb for model development")
print("2. Run 03_anomaly_detection_dev.ipynb for anomaly detection")
print("3. Run 04_evaluation_report.ipynb for full evaluation")

EDA SUMMARY REPORT
Total Records........................... 17924554
Date Range.............................. 2019-05-01 00:00:00 to 2021-10-31 23:57:00
Duration (days)......................... 914
Zones................................... 2
Missing Values.......................... 0
Duplicates.............................. 0
Memory (MB)............................. 2830.627305984497

✓ EDA Complete!

Next Steps:
1. Run 02_demand_forecasting_dev.ipynb for model development
2. Run 03_anomaly_detection_dev.ipynb for anomaly detection
3. Run 04_evaluation_report.ipynb for full evaluation
